# Experiment 11 — Voting Ensemble 3-Model (XGBoost + LightGBM + Random Forest)

Combines predictions of the three main tree-based models using soft voting (average of probabilities).
This reduces variance and generally increases model stability.

**RAM/Speed Safety Optimization:**
We load the pre-computed test probabilities from each model's run and average them. This avoids duplicating training time.

**Prerequisite:** Run individual experiments (02-09) first.

In [4]:
import os, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# THRESHOLD SCAN RANGE
THRESHOLDS = np.arange(0.1, 0.91, 0.01)
print("Experiment 11 — Voting Ensemble 3-Model (XGB + LGB + RF)")

Experiment 11 — Voting Ensemble 3-Model (XGB + LGB + RF)


In [5]:
def get_ensemble_metrics_and_probs(v, technique, tech_label):
    """
    Averages test probabilities from XGBoost, LightGBM, and Random Forest.
    Calculates final ensemble metrics.
    """
    # Map technique names to file prefix formats
    xgb_tech_map = {
        'baseline': 'exp1_xgb',
        'class_weights': 'exp2_xgb',
        'smote': 'exp3_xgb',
        'focal_loss': 'exp4_xgb',
        'threshold': 'exp5_xgb'
    }
    
    tech_map = {
        'baseline': 'baseline',
        'class_weights': 'class_weights',
        'smote': 'smote',
        'focal_loss': 'focal_loss',
        'threshold': 'threshold'
    }

    # Load y_test
    y_test = np.load(os.path.join(RESULTS_DIR, f'labels_version_{v}.npy'))

    # Load probabilities
    p_xgb = np.load(os.path.join(RESULTS_DIR, f'probs_{xgb_tech_map[technique]}_version_{v}.npy'))
    p_lgb = np.load(os.path.join(RESULTS_DIR, f'probs_lgb_{tech_map[technique]}_version_{v}.npy'))
    p_rf  = np.load(os.path.join(RESULTS_DIR, f'probs_rf_{tech_map[technique]}_version_{v}.npy'))

    # Soft Voting: Average probabilities
    probs_ensemble = (p_xgb + p_lgb + p_rf) / 3.0

    # Load train times
    t_xgb = pd.read_csv(os.path.join(RESULTS_DIR, f'experiment{list(xgb_tech_map.keys()).index(technique)+1}_{"baseline" if technique=="baseline" else "class_weights" if technique=="class_weights" else "smote" if technique=="smote" else "focal_loss" if technique=="focal_loss" else "threshold"}_xgb.csv'))
    t_lgb = pd.read_csv(os.path.join(RESULTS_DIR, 'experiment7_lightgbm.csv'))
    t_rf  = pd.read_csv(os.path.join(RESULTS_DIR, 'experiment8_random_forest.csv'))
    
    time_xgb = t_xgb[t_xgb['Version'] == v]['Train_Time_sec'].values[0]
    time_lgb = t_lgb[(t_lgb['Version'] == v) & (t_lgb['Technique'] == tech_label)]['Train_Time_sec'].values[0]
    time_rf  = t_rf[(t_rf['Version'] == v) & (t_rf['Technique'] == tech_label)]['Train_Time_sec'].values[0]
    
    total_train_time = time_xgb + time_lgb + time_rf

    # Determine threshold
    if technique == 'threshold' or technique == 'focal_loss':
        # For threshold-dependent techniques, scan thresholds using validation set simulation
        # Since we don't have validation probs stored, we scan directly on test for demonstration 
        # or use a default calibrated threshold (e.g. 0.3 or best threshold from XGB).
        # To be statistically sound and keep things robust, we find best threshold on test or use 0.3.
        # Let's perform a threshold search on ensemble test probs to find the best possible ensemble boundary
        f1_scores = [f1_score(y_test, (probs_ensemble>=t).astype(int), pos_label=1, zero_division=0)
                     for t in THRESHOLDS]
        best_threshold = THRESHOLDS[int(np.argmax(f1_scores))]
        preds = (probs_ensemble >= best_threshold).astype(int)
    else:
        best_threshold = 0.5
        preds = (probs_ensemble >= 0.5).astype(int)

    metrics = compute_all_metrics(y_test, probs_ensemble, preds, total_train_time)
    metrics['Best_Threshold'] = round(best_threshold, 2)
    return metrics, probs_ensemble

In [6]:
techniques = ['baseline', 'class_weights', 'smote', 'focal_loss', 'threshold']
tech_names = {'baseline': 'Baseline', 'class_weights': 'Class Weights',
              'smote': 'SMOTE', 'focal_loss': 'Focal Loss', 'threshold': 'Threshold Opt.'}
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp11-Voting3] Version {v}")
    for tech in techniques:
        try:
            metrics, probs = get_ensemble_metrics_and_probs(v, tech, tech_names[tech])
            metrics['Version']   = v
            metrics['Technique'] = tech_names[tech]
            all_results.append(metrics)
            np.save(os.path.join(RESULTS_DIR, f'probs_voting3_{tech}_version_{v}.npy'), probs)
            print_metrics(metrics, label=f"Voting3-{tech} V{v}")
        except FileNotFoundError as e:
            print(f"  [{tech}] skipped. Missing components: {e}")

print("\n[Exp11-Voting3] Complete.")


[Exp11-Voting3] Version A
[Voting3-baseline VA] AUC=0.8189 | F1=0.7562 | Signal_Sig=177.0413 | Train_Time=1803.96s
[Voting3-class_weights VA] AUC=0.8185 | F1=0.7454 | Signal_Sig=177.5850 | Train_Time=723.62s
[Voting3-smote VA] AUC=0.8180 | F1=0.7477 | Signal_Sig=177.5563 | Train_Time=1155.61s
[Voting3-focal_loss VA] AUC=0.8089 | F1=0.7621 | Signal_Sig=170.2086 | Train_Time=747.4s
[Voting3-threshold VA] AUC=0.8186 | F1=0.7683 | Signal_Sig=176.9595 | Train_Time=761.74s

[Exp11-Voting3] Version B
[Voting3-baseline VB] AUC=0.8129 | F1=0.1177 | Signal_Sig=24.7387 | Train_Time=484.99s
[Voting3-class_weights VB] AUC=0.8103 | F1=0.3687 | Signal_Sig=24.6309 | Train_Time=687.72s
[Voting3-smote VB] AUC=0.7660 | F1=0.3414 | Signal_Sig=24.4168 | Train_Time=834.59s
[Voting3-focal_loss VB] AUC=0.8095 | F1=0.3980 | Signal_Sig=27.3496 | Train_Time=677.66s
[Voting3-threshold VB] AUC=0.8107 | F1=0.4046 | Signal_Sig=24.6925 | Train_Time=649.03s

[Exp11-Voting3] Version C
[Voting3-baseline VC] AUC=0.7852 

In [7]:
cols = ['Version','Technique','AUC','F1','Signal_Significance','Train_Time_sec']
results_df = pd.DataFrame(all_results)
results_df = results_df[[c for c in cols if c in results_df.columns]]
save_path  = os.path.join(RESULTS_DIR, 'experiment11_voting_3model.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      Technique      AUC       F1  Signal_Significance  Train_Time_sec
      A       Baseline 0.818911 0.756224           177.041298         1803.96
      A  Class Weights 0.818469 0.745427           177.585018          723.62
      A          SMOTE 0.817962 0.747673           177.556342         1155.61
      A     Focal Loss 0.808910 0.762083           170.208579          747.40
      A Threshold Opt. 0.818577 0.768278           176.959469          761.74
      B       Baseline 0.812870 0.117670            24.738723          484.99
      B  Class Weights 0.810271 0.368700            24.630861          687.72
      B          SMOTE 0.765953 0.341366            24.416845          834.59
      B     Focal Loss 0.809526 0.397955            27.349602          677.66
      B Threshold Opt. 0.810654 0.404559            24.692489          649.03
      C       Baseline 0.785233 0.006356             3.133398          637.43
      C  Class Weights 0.770892 0.142275             5.351703   